# Cross-talk screen: rank a list of candidate target genes against one core (disease-causal) gene

**Inputs: `CORE` (disease-causal gene) and `TARGETS` (a list of candidate target genes).** For every candidate the same local cross-talk
analysis as `crosstalk_research.ipynb` is run (walk of CORE with the candidate removed, walk of the candidate with CORE removed), and each
candidate is scored on independent axes and ranked:

| column | meaning | reads as |
|---|---|---|
| `interference_score` | Σ edge weight / (1 + proximity) over edges from the candidate's pathway into the CORE pathway | how much, and how close, the candidate acts on CORE's pathway |
| `tf_enrichment`, `tf_p` | observed / expected shared TFs (hypergeometric) | is the overlap more than a hub would give by chance |
| `dependency` | 1 − \|D(CORE without candidate)\| / \|D(CORE)\| | share of CORE's pathway that *needs* the candidate (bypass question) |
| `net_direction` | (syn − ant) / (syn + ant) over interference edges (`syn_edges`, `ant_edges`) and shared endpoints (`syn_endpoints`, `ant_endpoints`) | +1 same direction as CORE, −1 opposes CORE (a property, not part of the rank) |
| `baseline_z`, `baseline_pct` | z / percentile vs degree-matched random genes against the same CORE | is it more than any gene of similar connectivity |
| `composite` | mean percentile rank of interference, enrichment (if p ≤ 0.5) and dependency | the ranking |

Walks only (no DepMap / STRING), so 50 candidates take a few minutes. Run `crosstalk_research.ipynb` on the top hits for residues and views.

## 1. Parameters

Database locations come from the YAML config (`config.example.yaml` → copy to `target_net.yaml`); parameters here only override it.

In [ ]:
CORE = "FGFR3"         # core / disease-causal gene
TARGETS = """
HSP90AA1
AKT1
MAPK1
GSK3B
MTOR
EGFR
"""                    # candidate target genes: plain text (newlines / commas / spaces), a Python list, or a path to a text/CSV file (first column)
DEMO = False           # False = your databases from target_net.yaml; True = offline curated demo graph
CONFIG = None          # None = find target_net.yaml in this folder or any parent (project root), else ~/.target_net.yaml; or give a path
TISSUE = None
MAX_DEPTH = MIN_SCORE = HUB_PENALTY = None   # None = values from the YAML crosstalk: section (local walks, default depth 3 / min_score 0.3)
N_RANDOM = 20          # degree-matched random genes for the baseline (0 = skip)
TARGET_NET_DIR = ".."

## 2. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath(TARGET_NET_DIR))
from targetnet.api import screen_crosstalk, screen_figure, research_crosstalk, summarize_crosstalk, figure
from targetnet.plots import use_cdn_renderer
use_cdn_renderer()   # plotly.js from CDN -> small .ipynb files
import pandas as pd
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 180)
OUT_ROOT = os.path.join(TARGET_NET_DIR, "runs")   # runs/<YYYYMMDD_HHMMSS>_crosstalk_screen_<CORE>_<n>targets/

## 3. Run the screen

In [ ]:
# One cross-talk analysis per candidate, then the baseline (random genes of similar connectivity), then the ranking
s = screen_crosstalk(CORE, TARGETS, config=CONFIG, demo=DEMO, tissue=TISSUE, max_depth=MAX_DEPTH, min_score=MIN_SCORE,
                     hub_penalty=HUB_PENALTY, n_random=N_RANDOM, out=OUT_ROOT)
print("outputs:", s["run_dir"])

## 4. Ranking table
Sorted by `composite`; `baseline_z` breaks ties. `top_edges` previews the strongest interference edges (syn / ant / mix).

In [ ]:
# Ranked table. syn/ant_edges = interference edges; syn/ant_endpoints = shared TFs/genes (the counts research_crosstalk prints)
tab = s["table"]
tab[["rank","target","interference_score","n_interference","shared_nodes","tf_enrichment","tf_p","dependency","syn_edges","ant_edges","syn_endpoints","ant_endpoints","net_direction","baseline_z","baseline_pct","composite","top_edges"]]

## 5. Screen view
x = interference score, y = TF-overlap enrichment, size = dependency, colour = direction (teal synergistic, brown antagonistic). Top-right and large = strong, mechanistically close cross-talk; hover for the numbers.

In [ ]:
screen_figure(s).show()

## 6. Interference edges behind the top candidates
The edges that produced the scores (top 5 per candidate): which node of the candidate's pathway acts on which node of the CORE pathway, with residue and sign agreement.

In [ ]:
e = s["edges"]
e[e.target.isin(tab.target.head(5))][["target","rank","from_node","to_node","hops_goi_to_from","hops_causal_to_to","mechanism","residue","net_effect_of_goi_on_target","causal_effect_on_target","agreement"]] if len(e) else print("no interference edges for any candidate at these walk settings")

## 7. Full analysis of one hit
Take the best candidate into the detailed cross-talk analysis (three-lane view, shared nodes, endpoint agreement). Change `HIT` to any gene in the table.

In [ ]:
HIT = tab.target.iloc[0] if len(tab) else None
if HIT:
    x = research_crosstalk(core=CORE, target=HIT, config=CONFIG, demo=DEMO, tissue=TISSUE, out=OUT_ROOT)
    print(summarize_crosstalk(x))
    figure(x["views"]["crosstalk"]).show()

## 8. How to interpret
1. **Rank is relative.** `composite` orders the candidates you gave; it says nothing about absolute strength. `baseline_z` does: a candidate with z > 2 interferes more with CORE's pathway than random genes of the same connectivity; z ≈ 0 means "what any gene of that size would show".
2. **Interference vs dependency are different questions.** High `interference_score` with low `dependency` = the candidate *modulates* CORE's pathway from outside (combination-therapy logic). High `dependency` = CORE's output *runs through* the candidate (inhibiting the candidate blocks CORE; bypass logic). Both high = the candidate sits on CORE's main route and also feeds it from the side.
3. **Direction decides the use.** `net_direction` < 0 (antagonistic): inhibiting the candidate would *release* CORE's pathway — a liability if CORE is oncogenic, an opportunity if CORE is a tumour suppressor. > 0 (synergistic): inhibiting the candidate also dampens CORE's output. Mixed near 0 with many edges = context-dependent; look at the edges, not the average.
4. **Enrichment ≈ 0 with many edges** means the candidate touches CORE's intermediates but not its transcriptional output; the effect may be post-translational (stability, localisation) rather than on gene expression.
5. **Skipped candidates** (printed during the run) had no outgoing signed edge in the graph — a database gap (see `signor-lookup`, the OmniPath gap filler and seed rescue), not evidence of no cross-talk.
6. Validate: the top edges carry residues; phosphoproteomics after inhibiting the candidate (in a cell model where CORE is active) tests them directly.